# Saluki for Gluc 3′ UTR design

[Notebook](saluki-gluc-3utr.ipynb) · [Installation](../docs/installation.md)

Optimize a 150-nt 3′ UTR for Saluki-predicted mRNA stability. The HBB 5′ UTR and Gluc CDS are fixed. Saluki scores the full single-exon transcript.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

from pathlib import Path
from tempfile import mkdtemp

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, random_seed_sequences, select_candidates

utr5 = "ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC"
cds = (
    "ATGGGAGTCAAAGTTCTGTTTGCCCTGATCTGCATCGCTGTGGCCGAGGCCAAGCCCACCGAGAACAACGAA"
    "GACTTCAACATCGTGGCCGTGGCCAGCAACTTCGCGACCACGGATCTCGATGCTGACCGCGGGAAGTTGCCC"
    "GGCAAGAAGCTGCCGCTGGAGGTGCTCAAAGAGATGGAAGCCAATGCCCGGAAAGCTGGCTGCACCAGGGGC"
    "TGTCTGATCTGCCTGTCCCACATCAAGTGCACGCCCAAGATGAAGAAGTTCATCCCAGGACGCTGCCACACC"
    "TACGAAGGCGACAAAGAGTCCGCACAGGGCGGCATAGGCGAGGCGATCGTCGACATTCCTGAGATTCCTGGG"
    "TTCAAGGACTTGGAGCCCATGGAGCAGTTCATCGCACAGGTCGATCTGTGTGTGGACTGCACAACTGGCTGC"
    "CTCAAAGGGCTTGCCAACGTGCAGTGTTCTGACCTGCTCAAGAAGTGGCTGCCGCAACGCTGTGCGACCTTT"
    "GCCAGCAAGATCCAGGGCCAGGTGGACAAGATCAAGGGGGCCGGTGGTGACTAA"
)
length = 150
prefix = utr5 + cds

device = "cuda" if torch.cuda.is_available() else "cpu"
saluki = load_model(
    "saluki", device=device,
    cds_start=len(utr5), cds_end=len(prefix),
    exon_lengths=[len(prefix) + length],
)

indices = torch.tensor(["ACGT".index(base) for base in prefix])
fixed = torch.nn.functional.one_hot(indices, num_classes=4)


class UTRScorer:
    task_names = saluki.task_names
    device = saluki.device

    def score(self, sequences):
        return saluki.score([prefix + sequence for sequence in sequences])

    def score_soft(self, probabilities):
        fixed_batch = fixed.to(probabilities).expand(len(probabilities), -1, -1)
        transcripts = torch.cat([fixed_batch, probabilities], dim=1)
        return saluki.score_soft(transcripts)


model = UTRScorer()
run = Path(mkdtemp(prefix="motifsift-example-"))




## 2. Generate candidates

Start with 100 random 3′ UTRs. Each of the eleven optimizers retains 100 candidates at 90% of the observed gain. The saved tables come from the full Gluc run with these starting sequences and the same Saluki checkpoint. Reruns use the current MotifSift defaults.



In [2]:
seeds = random_seed_sequences(length, 100, seed=42)
designs = generate_designs(
    model, length=length, initial=seeds, objective="maximize",
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,0.976557,1.002071
basesearch,100,0.921398,0.976028
beam_search,100,1.017275,1.089254
cross_entropy_method,100,0.925680,0.980721
den,100,0.987461,1.011349
fast_seqprop,100,0.965313,1.011965
genetic_algorithm,100,0.969405,0.999111
ism_greedy,100,0.907591,1.009200
ledidi,100,0.812605,0.935514




## 3. Discover motifs

Analyze the 3′ UTR in transcript orientation.



In [3]:
motifs = call_motifs(designs, model, reverse_complement=False, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
motif_id,,,,
1-UUAAUAWW,1-UUAAUAWW,539,515,737.0
2-CCCUAUUAR,2-CCCUAUUAR,183,164,229.0
4-ACCAMUAAAU,4-ACCAMUAAAU,269,245,609.0
5-RCWCCCCUAUA,5-RCWCCCCUAUA,154,148,912.0
6-CAAWAUAGGC,6-CAAWAUAGGC,95,94,58.0
10-YUCCCCCCUA,10-YUCCCCCCUA,100,100,1310.0
15-AAAUAAUUARW,15-AAAUAAUUARW,128,126,50.0
25-UUAAUUAACCC,25-UUAAUUAACCC,53,53,307.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,CTAATAAATATCTTTTACTGAGTTTACCCCCGCCCTGTGCTCGCCC...,1.089254,1-UUAAUAWW;2-CCCUAUUAR;5-RCWCCCCUAUA
1,2,TTATTACTAACTATCTTTTCTTACTCCACTTCCCTTCACTTTTACC...,0.961090,10-YUCCCCCCUA
2,3,CCAATAATAATCCCCCCGCTTTGCACCCTCCCCGCGCCTCGCATCA...,1.007690,25-UUAAUUAACCC
3,4,CAATCCCAAATCAATACCCCTTATTAAAAGATTAAAGATCCTCGAG...,0.901135,4-ACCAMUAAAU
4,5,TTAATAATAATTAATAATAGTCTTGCTAGCTAGAAATTAGTCTTAG...,0.955507,1-UUAAUAWW;15-AAAUAAUUARW
5,6,CCCCTAATAATTCTCCTAGGTACAATTGGGCTAAATAGGCAATAAA...,0.935617,2-CCCUAUUAR;6-CAAWAUAGGC
6,7,CCCCCCCCGCCCCCCCCCCCCCGCTATACCCCTGCGTGTTTCCAGC...,0.953220,5-RCWCCCCUAUA
7,8,GCCCGCGCTGCCCCTATAGTCCGCGCGCCCCCCCCCCCCCTCCCCT...,0.959950,4-ACCAMUAAAU;5-RCWCCCCUAUA;10-YUCCCCCCUA;25-UU...
8,9,ACGACGATTGGTACCCCCTATGCGTTTTTCCCGCTTATACGTGTGT...,1.020161,6-CAAWAUAGGC;10-YUCCCCCCUA
9,10,TAATTATTACCCTTTGCTCATCCTTAATCCCATGATTCCCCCCACC...,0.912272,2-CCCUAUUAR;10-YUCCCCCCUA
